# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthGowda1718/ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule: flag a page as needing an engagement fix if its CTR is meaningfully below the median CTR for pages at the same rank position (position_tier) — specifically, below 60% of that tier's median CTR. This uses the confirmed signal from the signal audit: position clearly predicts expected CTR, so a large gap from that expectation is the actionable anomaly.

Reason codes the rule can output:
- LOW_CTR_FOR_POSITION: CTR is well below the tier median — the core trigger.
- STRONG_POSITION_WASTED: page ranks well (position_tier is a top tier) but still has low CTR — high-value opportunity.
- LOW_VOLUME_LOW_PRIORITY: page has low search_volume, so even if flagged, it's lower priority for review time.
- INSUFFICIENT_DATA: page has very few days_with_impressions, so the CTR signal isn't reliable yet — flagged for monitoring, not immediate action.

In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/jayanthGowda1718/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

expected_ctr = df.groupby("position_tier")["ctr"].transform("median")
df["ctr_gap_ratio"] = df["ctr"] / expected_ctr
df["needs_engagement_fix"] = (df["ctr_gap_ratio"] < 0.6).astype(int)

print(df["needs_engagement_fix"].value_counts())

needs_engagement_fix
0    18417
1    11583
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score every page by how far its CTR falls below expectation, then rank the whole dataset by that score so the content team gets a prioritized queue — worst gap first — along with a reason code explaining why each page was flagged.

In [6]:
import numpy as np

def assign_reason_code(row):
    if row["days_with_impressions"] < 10:
        return "INSUFFICIENT_DATA"
    if row["ctr_gap_ratio"] < 0.6 and row["position_tier"] in ["top", "1-3", "1"]:  # adjust to real tier labels
        return "STRONG_POSITION_WASTED"
    if row["ctr_gap_ratio"] < 0.6 and row["search_volume"] < df["search_volume"].median():
        return "LOW_VOLUME_LOW_PRIORITY"
    if row["ctr_gap_ratio"] < 0.6:
        return "LOW_CTR_FOR_POSITION"
    return "OK"

df["reason_code"] = df.apply(assign_reason_code, axis=1)

# Baseline score: lower ctr_gap_ratio = worse gap = higher priority
df["baseline_score"] = 1 - df["ctr_gap_ratio"].clip(upper=1)

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "avg_position", "position_tier", "ctr", "ctr_gap_ratio",
     "baseline_score", "reason_code"]
]

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

queue.head(20)

,content_id,avg_position,position_tier,ctr,ctr_gap_ratio,baseline_score,reason_code
11614,content_eedda1271288,12.0,striking,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11615,content_94f382daa1c3,33.7,page_3_5,0.0,0.0,1.0,LOW_VOLUME_LOW_PRIORITY
11618,content_4bbc3329299f,12.7,striking,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11587,content_9ea18c92ffa4,18.9,striking,0.0,0.0,1.0,LOW_VOLUME_LOW_PRIORITY
11591,content_1fe326350ef1,10.4,striking,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11592,content_806c02f4994a,5.9,page_1,0.0,0.0,1.0,INSUFFICIENT_DATA
11593,content_fbbd736e7c53,6.0,page_1,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11597,content_231fa9ab4593,33.4,page_3_5,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11621,content_12878065f88e,9.2,page_1,0.0,0.0,1.0,LOW_VOLUME_LOW_PRIORITY
11622,content_d5685a556222,3.8,page_1,0.0,0.0,1.0,INSUFFICIENT_DATA


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 flagged pages from the ranked queue: each gets a recommended action, its reason code, a confidence note based on how much data backs the flag, and a note on what evidence would prove the flag wrong.

General pattern: pages with reason_code = STRONG_POSITION_WASTED get highest confidence (good rank, low CTR is a strong, hard-to-explain-away signal) — action: review title/meta description immediately. Pages with reason_code = LOW_CTR_FOR_POSITION get medium confidence — action: review content relevance and SERP snippet. Pages with reason_code = INSUFFICIENT_DATA get low confidence — action: monitor for more data before committing review time; the flag could be wrong simply because the CTR estimate is noisy from few days of data. Pages with reason_code = LOW_VOLUME_LOW_PRIORITY are correctly flagged but low business impact — action: deprioritize behind higher-volume flags even if the gap is real.

Across all top 20, the flag would be wrong if: the page recently changed URLs or was redirected (breaking the CTR history), if there's a seasonal dip unrelated to the page itself, or if the position_tier assignment itself was stale relative to a recent ranking change.

In [7]:
# Pull the actual top 20 to inspect
queue.head(20)

,content_id,avg_position,position_tier,ctr,ctr_gap_ratio,baseline_score,reason_code
11614,content_eedda1271288,12.0,striking,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11615,content_94f382daa1c3,33.7,page_3_5,0.0,0.0,1.0,LOW_VOLUME_LOW_PRIORITY
11618,content_4bbc3329299f,12.7,striking,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11587,content_9ea18c92ffa4,18.9,striking,0.0,0.0,1.0,LOW_VOLUME_LOW_PRIORITY
11591,content_1fe326350ef1,10.4,striking,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11592,content_806c02f4994a,5.9,page_1,0.0,0.0,1.0,INSUFFICIENT_DATA
11593,content_fbbd736e7c53,6.0,page_1,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11597,content_231fa9ab4593,33.4,page_3_5,0.0,0.0,1.0,LOW_CTR_FOR_POSITION
11621,content_12878065f88e,9.2,page_1,0.0,0.0,1.0,LOW_VOLUME_LOW_PRIORITY
11622,content_d5685a556222,3.8,page_1,0.0,0.0,1.0,INSUFFICIENT_DATA


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: rows flagged with reason_code = INSUFFICIENT_DATA are the weakest — a large ctr_gap_ratio built on very few days of impressions is more noise than signal, and shouldn't be trusted with the same confidence as a page with a full 90-day history.

Leakage check: confirming the ranking itself doesn't use any of the excluded columns identified in the feature leakage check (w03_feature_leakage_check) — specifically no product flags (trend_direction, trend_pct, impression_tier) and no future-window fields beyond the 90-day baseline window.

In [8]:
used_columns = {"content_id", "avg_position", "position_tier", "ctr", "ctr_gap_ratio",
                 "baseline_score", "reason_code", "days_with_impressions", "search_volume"}

leakage_columns = {"trend_direction", "trend_pct", "impression_tier",
                    "clicks_90d", "engaged_sessions_90d", "scroll_events_90d", "scroll_rate"}

overlap = used_columns & leakage_columns
print("Leakage columns used in scoring:", overlap)

Leakage columns used in scoring: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.